In [12]:
import numpy as np
import matplotlib.pyplot as plt
K=5
e=0.6
T=20000

def rate_calculation_caseA(arm,instantaneous_snr):
  (j1,j2)= arm
  r = np.log(1+instantaneous_snr[0,j1]*10) + np.log(1+instantaneous_snr[1,j2]*10)
  return r

def rate_calculation_caseB(arm, instantaneous_snr):
  (j1,j2)= arm
  r1 = np.log(1+instantaneous_snr[0,j1]*10)
  r2 = np.log(1+instantaneous_snr[1,j2]*10)
  return r1, r2

def expected_log(center):
    F = lambda x: ((1 + 10*x) * np.log(1 + 10*x) - (1 + 10*x)) / 10 # antiderivative calculation
    return (F(center + e) - F(center - e)) / (2 * e)

# a function to find the true mean of the best arm in Case A
def true_mean(s,arms):
    mu_arms = [ expected_log(s[0, j1]) + expected_log(s[1, j2])
                for (j1, j2) in arms ]
    best_idx = np.argmax(mu_arms)

    mu_star = mu_arms[best_idx]
    best_arm = arms[best_idx]

    return mu_star, mu_arms, best_arm

def instance_dependent_bound_caseA(mu_star, mu_arms):

    deltas = [mu_star - mu for mu in mu_arms]
    bound = []

    for t in range(1, T + 1):
        bound_t = 0.0
        for delta in deltas:
            if delta > 0: #only suboptimal arms contribute to regret
               bound_t += 8 * (2 * np.log(9))**2 * np.log(t) / delta

        bound.append(bound_t)

    return bound ,deltas

# a function to find the true mean of the best arm in Case B
def true_mean_per_user(s, arms):
    mu_user = [[expected_log(s[i, j]) for j in range(K)] for i in range(2)]

    mu_arms = [mu_user[0][j1] + mu_user[1][j2] for (j1, j2) in arms]
    best_idx = np.argmax(mu_arms)

    mu_star = mu_arms[best_idx]
    best_arm = arms[best_idx]

    return mu_user, best_arm, mu_star

def instance_dependent_bound_caseB(mu1_star, mu2_star, mu1_arms, mu2_arms):

    deltas1 = [mu1_star - mu for mu in mu1_arms]
    deltas2 = [mu2_star - mu for mu in mu2_arms]
    bound = []

    for t in range(1, T + 1):
        bound_t = 0.0
        for delta in deltas1:
            if delta > 0:
                bound_t += 8 * (np.log(9))**2 * np.log(t) / delta

        for delta in deltas2:
            if delta > 0:
               bound_t += 8 * (np.log(9))**2 * np.log(t) / delta

        bound.append(bound_t)

    return bound, deltas1, deltas2


def case_A(mc_seed):
  # in this case our arms (which are the pairs of cahnnels) are 20 since we choose pairs of 5 channels and pairs of same channels are forbidden
  arms = [(j1,j2) for j1 in range(K) for j2 in range(K) if j1!=j2]
  N=[0]*len(arms) # we keep how many times each arm was played
  mi_hat=[0]*len(arms) # thats our estimate of the mean reward

  rng = np.random.default_rng(46)
  s = rng.uniform(0.8, 2.0, size=(2,K)) # we generate a 2x5 matrix that holds the values s_{i,j} which are the SNR parameters for each channel
  #print(s)
  mu_star, mu_arms, best_arm = true_mean(s,arms)

  bound_A, deltas = instance_dependent_bound_caseA(mu_star, mu_arms)
  #print(deltas)

  cum_regret = 0.0
  regret_samples = []
  regret_over_t = []
  cum_reward = 0.0
  cum_reward_per_round = []

  rng_play = np.random.default_rng(mc_seed)
  for t in range(1, T+1):

    instantaneous_snr = rng_play.uniform(s-e, s+e)
    # for the first 20 rounds we play each arm once
    if t <= len(arms):
      a = t-1
      arm = arms[a]
    else:
      ucb = np.array(mi_hat) +  2*np.log(9)*np.sqrt(2*np.log(T) / np.array(N))
      a = np.argmax(ucb) #  μi(hat)+ε
      arm = arms[a]

    # reward tracking
    r = rate_calculation_caseA(arm, instantaneous_snr)
    cum_reward += r
    cum_reward_per_round.append(cum_reward)

    instant_regret = mu_star - mu_arms[a]
    cum_regret += instant_regret

    regret_samples.append(cum_regret)
    regret_over_t.append(cum_regret / t)



    N[a]+=1
    mi_hat[a] = mi_hat[a] + (r-mi_hat[a])/N[a]

  return arms, N, regret_samples, regret_over_t, cum_reward_per_round , bound_A , deltas , mu_arms

def case_B(mc_seed):

  arms = [(j1,j2) for j1 in range(K) for j2 in range(K) if j1!=j2]
  N_arms = [0]*len(arms)   # how many times each arm was played

  '''
    we use a list of lists where,
    N_user[i][j] is how many times the i-th user used channel j
  '''
  N_user = [[0 for _ in range(K)] for _ in range(2)]

  '''
  # μ_{i,j} -> i-th user using j-th channel :

   [  [μ_{1,1},μ_{1,2},...,μ_{1,5}], [μ_{2,1},μ_{2,2},...,μ_{2,5}]  ]

  '''
  mu_hat_user = [[0.0 for _ in range(K)] for _ in range(2)]

  rng = np.random.default_rng(46)
  s = rng.uniform(0.8, 2.0, size=(2,K)) # we generate a 2x5 matrix that holds the values s_{i,j} which are the SNR parameters for each channel
  mu_user,best_arm,mu_star = true_mean_per_user(s,arms)
  mu1_star = mu_user[0][best_arm[0]]
  mu2_star = mu_user[1][best_arm[1]]
  bound_B, deltas1, deltas2 = instance_dependent_bound_caseB( mu1_star,mu2_star,mu_user[0],mu_user[1])

  cum_regret = 0.0
  regret_samples = []
  regret_over_t = []
  cum_reward = 0.0
  cum_reward_per_round = []

  arm_to_index = {arm:i for i,arm in enumerate(arms)}

  rng_play = np.random.default_rng(mc_seed)
  for t in range(1, T+1):

    instantaneous_snr = rng_play.uniform(s-e, s+e)
    # for the first 5 rounds we want to play each channel once for each user,else we end up dividing by zero, so
    # we need to play the arms : (0,1),(1,2),(2,3),(3,4),(4,0)

    if t <= 5:
      j1 = t-1
      j2 = t % K
      arm = (j1, j2)
      a = arm_to_index[arm]
    else:
      '''
       ucb_user[] is list of two list (2 x K) that holds the UCB value of each user-channel
       it looks like this: [ [UCB_1,0, UCB_1,1, ..., UCB_1,4],
                             [UCB_2,0, UCB_2,1, ..., UCB_2,4]
                            ]
      '''
      ucb_user = [[0.0 for _ in range(K)] for _ in range(2)] # in each round we need to clear the means of each user and calculate the renewed ones

      for i in range(2):
        for j in range(K):
          ucb_user[i][j] = mu_hat_user[i][j] + np.log(9)*np.sqrt(2*np.log(T) / N_user[i][j])

      # now we need to play the pair that has the biggest sum of optimistic estimate (UCB1 + UCB2)
      # in order to do that we save the sum : (UCB1 + UCB2)of each arm in a list ,
      # then the maximum element of that list will be the arm we choose to play
      arm_scores = []

      for idx, arm_candidate in enumerate(arms):
        (j1, j2) = arm_candidate
        score = ucb_user[0][j1] + ucb_user[1][j2]
        arm_scores.append((score, idx))

      best_score, best_a = max(arm_scores)
      arm = arms[best_a]
      a = best_a

    N_arms[a] += 1

    (j1,j2)=arm
    mu_played = mu_user[0][j1] + mu_user[1][j2]
    instant_regret = mu_star - mu_played

    cum_regret += instant_regret
    regret_samples.append(cum_regret)
    regret_over_t.append(cum_regret / t)

    r1, r2 = rate_calculation_caseB(arm, instantaneous_snr)
    r = r1 + r2

    cum_reward += r
    cum_reward_per_round.append(cum_reward)

    # update estimates for user1 on channel j1
    N_user[0][j1] += 1
    mu_hat_user[0][j1] = mu_hat_user[0][j1] + (r1 - mu_hat_user[0][j1]) / N_user[0][j1]

    # update estimates for user2 on channel j2
    N_user[1][j2] += 1
    mu_hat_user[1][j2] = mu_hat_user[1][j2] + (r2 - mu_hat_user[1][j2]) / N_user[1][j2]


  return arms, N_arms, regret_samples, regret_over_t, cum_reward_per_round , bound_B , deltas1 , deltas2 , mu_user


#----------------------- PLOTTING RESULTS used Chat here-----------------------------------------------------------
def plot_results(arms, N, regret_samples, regret_over_t, cum_reward_per_round, bound, title_prefix):

  # Cumulative Regret vs Bound A
  plt.figure()
  plt.plot(range(1, T + 1), np.maximum(regret_samples, 1e-8), label="Empirical cumulative regret")
  plt.plot(range(1, T + 1), np.maximum(bound, 1e-8), color='r', linestyle='--', label="Instance-dependent bound")
  plt.xlabel("t")
  plt.ylabel("Cumulative regret R(t)")
  plt.title(f"{title_prefix}: Cumulative Regret")
  plt.yscale("log")
  plt.legend()
  plt.savefig(f"{title_prefix}_Cumulative_Regret_with_Bound.jpg", dpi=300, bbox_inches='tight')
  plt.close()


  # Cumulative Regret
  plt.figure()

  plt.plot(range(1, T + 1), regret_samples, label="Cumulative regret")

  plt.xlabel("t")
  plt.ylabel("Cumulative regret R(t)")
  plt.title(f"{title_prefix}: Cumulative Regret")
  plt.legend()
  plt.savefig(f"{title_prefix}_Cumulative_Regret.jpg", dpi=300, bbox_inches='tight')
  plt.close()

  # Regret/T
  plt.figure()
  plt.plot(range(1, T+1), regret_over_t)
  plt.xlabel("Time")
  plt.ylabel("Regret/T")
  plt.title(f"{title_prefix}: Regret per Round (Regret/T)")
  plt.savefig(f"{title_prefix}_Regret_per_Round.jpg", dpi=300, bbox_inches='tight')
  plt.close()

  # Cumulative Reward
  plt.figure()
  plt.plot(range(1, T + 1), cum_reward_per_round)
  plt.xlabel("t")
  plt.ylabel("Cumulative reward")
  plt.title(f"{title_prefix}: Total Reward Over Time")
  plt.savefig(f"{title_prefix}_Total_Reward.jpg", dpi=300, bbox_inches='tight')
  plt.close()

  # Arm Plays
  order = np.argsort(N)[::-1]
  N_sorted = [N[i] for i in order]
  arms_sorted = [str(arms[i]) for i in order]

  plt.figure(figsize=(8, 6))
  plt.barh(range(len(arms_sorted)), N_sorted)
  plt.yticks(range(len(arms_sorted)), arms_sorted)
  plt.xlabel("Times played (log scale)")
  plt.title(f"{title_prefix}: Arm Plays")
  plt.xscale("log")

  for y, val in enumerate(N_sorted):
      if val > 0:
        plt.text(val, y, f" {val}", va="center")

  plt.gca().invert_yaxis()
  plt.tight_layout()
  plt.savefig(f"{title_prefix}_Arm_Plays.jpg", dpi=300, bbox_inches='tight')
  plt.close()

def compare_cases(regret_A, regret_B,
                  regret_over_t_A, regret_over_t_B,
                  reward_A, reward_B,
                  bound_A, bound_B,):

  # Cumulative Regret
  plt.figure()
  plt.plot(range(1, T+1), regret_A, label="Case A")
  plt.plot(range(1, T+1), regret_B, label="Case B")
  plt.xlabel("t")
  plt.ylabel("Cumulative regret R(t)")
  plt.title("Cumulative Regret: Case A vs Case B")
  plt.legend()
  plt.savefig("Comparison_Cumulative_Regret.jpg", dpi=300, bbox_inches='tight')
  plt.close()

  # Cumulative Regret + Bounds
  plt.figure()
  plt.plot(range(1, T+1), np.maximum(regret_A, 1e-8), label="Case A empirical regret")
  plt.plot(range(1, T+1), np.maximum(regret_B, 1e-8), label="Case B empirical regret")
  plt.plot(range(1, T+1), np.maximum(bound_A, 1e-8), linestyle='--', label="Case A bound")
  plt.plot(range(1, T+1), np.maximum(bound_B, 1e-8), linestyle='--', label="Case B bound")

  plt.xlabel("t")
  plt.ylabel("Cumulative regret R(t)")
  plt.title("Cumulative Regret and Bounds: Case A vs Case B")
  plt.yscale("log")
  plt.legend()

  plt.savefig("Comparison_Cumulative_Regret_with_Bounds.jpg", dpi=300, bbox_inches='tight')
  plt.close()

  # Regret per Round
  plt.figure()
  plt.plot(range(1, T+1), regret_over_t_A, label="Case A")
  plt.plot(range(1, T+1), regret_over_t_B, label="Case B")
  plt.xlabel("t")
  plt.ylabel("Regret/T")
  plt.title("Regret per Round: Case A vs Case B")
  plt.legend()
  plt.savefig("Comparison_Regret_per_Round.jpg", dpi=300, bbox_inches='tight')
  plt.close()

  # Cumulative Reward
  plt.figure()
  plt.plot(range(1, T+1), reward_A, label="Case A")
  plt.plot(range(1, T+1), reward_B, label="Case B")
  plt.xlabel("t")
  plt.ylabel("Cumulative reward")
  plt.title("Cumulative Reward: Case A vs Case B")
  plt.legend()
  plt.savefig("Comparison_Cumulative_Reward.jpg", dpi=300, bbox_inches='tight')
  plt.close()

#--------------------------------------------------------------------------------------------------------

if __name__ == "__main__":

  # we run just to print the matrices of mean rewards and Deltas
    arms_A, N_arms_A, regret_samples_A, regret_over_t_A, cum_reward_per_round_A, bound_A, deltas,mu_arms_A = case_A(47)
    arms_B, N_arms_B, regret_samples_B, regret_over_t_B, cum_reward_per_round_B, bound_B, deltas1, deltas2 ,mu_user_B = case_B(47)

  #---------------------------- chated below...to have prints easy on the eye ---------------------------
    print("\n" + "="*55)
    print("ENVIRONMENT PARAMETERS")
    print("="*55)

    print("\nCase A – Arm Mean Rewards (μ_i) and Gaps (Δ_i)")
    print("-"*55)

    for arm, mu, delta in zip(arms_A, mu_arms_A, deltas):
        print(f"Arm {str(arm):>6} :  μ_i = {float(mu):8.6f}    Δ_i = {float(delta):8.6f}")

    print("\n" + "="*55)
    print("Case B – Channel Means and Gaps per User")
    print("="*55)

    print("\nUser 1")
    print("-"*30)
    for j in range(K):
        print(f"Channel {j} :  μ_1,{j} = {float(mu_user_B[0][j]):8.6f}    Δ_1,{j} = {float(deltas1[j]):8.6f}")

    print("\nUser 2")
    print("-"*30)
    for j in range(K):
        print(f"Channel {j} :  μ_2,{j} = {float(mu_user_B[1][j]):8.6f}    Δ_2,{j} = {float(deltas2[j]):8.6f}")

    print("\n" + "="*55 + "\n")
  #---------------------------------------------------------------------------------------------------------
    MC=10

    regret_A_all = np.zeros((MC, T))
    regret_over_t_A_all = np.zeros((MC, T))
    reward_A_all = np.zeros((MC, T))

    regret_B_all = np.zeros((MC, T))
    regret_over_t_B_all = np.zeros((MC, T))
    reward_B_all = np.zeros((MC, T))


    for m in range(MC):

        arms_A, N_arms_A, regret_samples_A, regret_over_t_A, cum_reward_per_round_A, bound_A,_,_ = case_A(m)
        arms_B, N_arms_B, regret_samples_B, regret_over_t_B, cum_reward_per_round_B ,bound_B,_,_,_= case_B(m)

        '''
           All below follow this style -> reward_A_all= [

                                                        [run1_t1, run1_t2, ..., run1_tT],
                                                        [run2_t1, run2_t2, ..., run2_tT],
                                                        ...
                                                        [runMC_t1, runMC_t2, ..., runMC_tT]

                                                      ]

        '''
        regret_A_all[m, :] = regret_samples_A
        regret_over_t_A_all[m, :] = regret_over_t_A
        reward_A_all[m, :] = cum_reward_per_round_A

        regret_B_all[m, :] = regret_samples_B
        regret_over_t_B_all[m, :] = regret_over_t_B
        reward_B_all[m, :] = cum_reward_per_round_B



    avg_regret_A = np.mean(regret_A_all, axis=0)
    avg_regret_over_t_A = np.mean(regret_over_t_A_all, axis=0)
    avg_reward_A = np.mean(reward_A_all, axis=0)

    avg_regret_B = np.mean(regret_B_all, axis=0)
    avg_regret_over_t_B = np.mean(regret_over_t_B_all, axis=0)
    avg_reward_B = np.mean(reward_B_all, axis=0)

    # Plot reasults
    plot_results(arms_A, N_arms_A, avg_regret_A, avg_regret_over_t_A, avg_reward_A, bound_A, "UCB Case A")
    plot_results(arms_B, N_arms_B, avg_regret_B, avg_regret_over_t_B, avg_reward_B, bound_B, "UCB Case B")

    # cases comparison
    compare_cases(avg_regret_A, avg_regret_B,
                  avg_regret_over_t_A, avg_regret_over_t_B,
                  avg_reward_A, avg_reward_B,
                  bound_A,bound_B)



ENVIRONMENT PARAMETERS

Case A – Arm Mean Rewards (μ_i) and Gaps (Δ_i)
-------------------------------------------------------
Arm (0, 1) :  μ_i = 5.757043    Δ_i = 0.122719
Arm (0, 2) :  μ_i = 5.862201    Δ_i = 0.017561
Arm (0, 3) :  μ_i = 5.663079    Δ_i = 0.216683
Arm (0, 4) :  μ_i = 5.843603    Δ_i = 0.036159
Arm (1, 0) :  μ_i = 4.463100    Δ_i = 1.416662
Arm (1, 2) :  μ_i = 5.114944    Δ_i = 0.764818
Arm (1, 3) :  μ_i = 4.915821    Δ_i = 0.963940
Arm (1, 4) :  μ_i = 5.096345    Δ_i = 0.783417
Arm (2, 0) :  μ_i = 4.701061    Δ_i = 1.178701
Arm (2, 1) :  μ_i = 5.247746    Δ_i = 0.632015
Arm (2, 3) :  μ_i = 5.153783    Δ_i = 0.725979
Arm (2, 4) :  μ_i = 5.334306    Δ_i = 0.545455
Arm (3, 0) :  μ_i = 5.014905    Δ_i = 0.864857
Arm (3, 1) :  μ_i = 5.561591    Δ_i = 0.318171
Arm (3, 2) :  μ_i = 5.666749    Δ_i = 0.213013
Arm (3, 4) :  μ_i = 5.648151    Δ_i = 0.231611
Arm (4, 0) :  μ_i = 5.227918    Δ_i = 0.651844
Arm (4, 1) :  μ_i = 5.774603    Δ_i = 0.105158
Arm (4, 2) :  μ_i = 5.8797